## Data Preparation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import linregress

# ── LOAD ──────────────────────────────────────────────────────────────────────
nodes = pd.read_csv("../mc1_csv/mc1_nodes.csv")
edges = pd.read_csv("../mc1_csv/mc1_edges.csv")

persons   = nodes[nodes["Node Type"] == "Person"].copy()
songs     = nodes[nodes["Node Type"] == "Song"].copy()
albums    = nodes[nodes["Node Type"] == "Album"].copy()
groups    = nodes[nodes["Node Type"] == "MusicalGroup"].copy()
labels    = nodes[nodes["Node Type"] == "RecordLabel"].copy()
all_works = pd.concat([songs, albums])

all_works["release_date"]   = pd.to_numeric(all_works["release_date"],   errors="coerce")
all_works["notoriety_date"] = pd.to_numeric(all_works["notoriety_date"], errors="coerce")

performer_of = edges[edges["Edge Type"] == "PerformerOf"]
composer_of  = edges[edges["Edge Type"] == "ComposerOf"]
lyricist_of  = edges[edges["Edge Type"] == "LyricistOf"]
in_style_of  = edges[edges["Edge Type"] == "InStyleOf"]
member_of    = edges[edges["Edge Type"] == "MemberOf"]
producer_of  = edges[edges["Edge Type"] == "ProducerOf"]
recorded_by  = edges[edges["Edge Type"] == "RecordedBy"]
distributed  = edges[edges["Edge Type"] == "DistributedBy"]
interpolates = edges[edges["Edge Type"] == "InterpolatesFrom"]
cover_of     = edges[edges["Edge Type"] == "CoverOf"]
lyrical_ref  = edges[edges["Edge Type"] == "LyricalReferenceTo"]
samples      = edges[edges["Edge Type"] == "DirectlySamples"]

all_influence = pd.concat([in_style_of, interpolates, cover_of, lyrical_ref, samples])

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
CURRENT_YEAR  = 2040
SAILOR_ID     = 17255
RECENT_WINDOW = 10    # widened from 3 — covers 2035–2040 for recent notable works
RECENT_COLLAB = 5    # last 5 years for collab activity = 2035–2040
GENRE_WINDOW  = 10   # separate wider window for hot genre detection = 2030–2040
MAX_DEBUT_AGE = 15   # hard filter: debuted 2025 or later

print("=" * 60)
print("VAST MC1 — Rising Star Pipeline")
print("=" * 60)
print(f"Current year : {CURRENT_YEAR}")
print(f"Recent window: last {RECENT_WINDOW} years ({CURRENT_YEAR - RECENT_WINDOW}–{CURRENT_YEAR})")
print(f"Genre window : last {GENRE_WINDOW} years ({CURRENT_YEAR - GENRE_WINDOW}–{CURRENT_YEAR})")
print(f"Debut filter : debuted {CURRENT_YEAR - MAX_DEBUT_AGE} or later")
print()

# ── BASE: artist → works lookup ───────────────────────────────────────────────
artist_works_df = performer_of.merge(
    all_works[["id", "genre", "release_date", "notable", "notoriety_date"]],
    left_on="target", right_on="id", how="left"
).rename(columns={"source": "artist_id"})

artist_works_df["release_date"]   = pd.to_numeric(artist_works_df["release_date"],   errors="coerce")
artist_works_df["notoriety_date"] = pd.to_numeric(artist_works_df["notoriety_date"], errors="coerce")

# Dictionary: artist_id → set of their work IDs (used for influence lookups)
work_ids_by_artist = performer_of.groupby("source")["target"].apply(set).to_dict()

# ── BASE CAREER METRICS ───────────────────────────────────────────────────────
metrics = artist_works_df.groupby("artist_id").agg(
    debut_year      = ("release_date",    "min"),
    latest_year     = ("release_date",    "max"),
    total_works     = ("id",              "count"),
    notable_works   = ("notable",         "sum"),
    first_notoriety = ("notoriety_date",  "min"),
    last_notoriety  = ("notoriety_date",  "max"),
    genre_diversity = ("genre",           "nunique"),
).reset_index()

metrics["notoriety_lag"] = metrics["first_notoriety"] - metrics["debut_year"]
metrics["career_span"]   = metrics["latest_year"] - metrics["debut_year"]

# ── HARD FILTER FLAGS ─────────────────────────────────────────────────────────
# Flag 1: must have at least one PerformerOf edge
performers_set        = set(performer_of["source"].unique())
metrics["is_performer"]     = metrics["artist_id"].isin(performers_set)

# Flag 2: debuted within the last MAX_DEBUT_AGE years
metrics["is_recent_debut"]  = metrics["debut_year"] >= (CURRENT_YEAR - MAX_DEBUT_AGE)

# ── SCORE PARAM 1: RECENT NOTABLE WORKS (last RECENT_WINDOW years) ────────────
# Stronger rising signal than total notable works — recency confirms current momentum
recent_notable = (
    artist_works_df[
        (artist_works_df["release_date"] >= (CURRENT_YEAR - RECENT_WINDOW)) &
        (artist_works_df["notable"] == True)
    ]
    .groupby("artist_id").size()
    .reset_index(name="recent_notable_works")
)
metrics = metrics.merge(recent_notable, on="artist_id", how="left")
metrics["recent_notable_works"] = metrics["recent_notable_works"].fillna(0)

print(f"Artists with recent notable works (last {RECENT_WINDOW}yr): "
      f"{(metrics['recent_notable_works'] > 0).sum()}")

# ── SCORE PARAM 2: OUTBOUND INFLUENCE ────────────────────────────────────────
# How many influence-type edges go FROM this artist's songs TO other songs
# Signals musical engagement and deliberate style-building
metrics["outbound_influence"] = metrics["artist_id"].map(
    lambda aid: all_influence[
        all_influence["source"].isin(work_ids_by_artist.get(aid, set()))
    ].shape[0]
)

# ── SCORE PARAM 3: NOTORIETY RECENCY ──────────────────────────────────────────
# Decays from 1.0 (charted this year) → 0.0 (charted 10+ years ago or never)
# Penalises one-hit wonders whose only chart hit was years ago
metrics["years_since_notoriety"]  = CURRENT_YEAR - metrics["last_notoriety"]
metrics["notoriety_recency_score"] = (
    1 - (metrics["years_since_notoriety"].clip(0, 10) / 10)
).fillna(0)

# ── SCORE PARAM 4: COLLABORATED WITH POPULAR ARTISTS ─────────────────────────
# Popular = top quartile by notable_works + Sailor Shift explicitly included
# Captures the "association effect" — appearing on a track with a star
# exposes you to their audience

popular_ids       = set(metrics[metrics["notable_works"] >= 3]["artist_id"])
popular_ids.add(SAILOR_ID)
print(f"Popular artist pool size: {len(popular_ids)}")

# Build (artist_a, artist_b, song_id, release_date) collab pair table
song_performers = performer_of.groupby("target")["source"].apply(list).reset_index()
shared_songs    = song_performers[song_performers["source"].apply(len) > 1]

collab_pairs = []
for _, row in shared_songs.iterrows():
    perfs   = row["source"]
    song_id = row["target"]
    for i in range(len(perfs)):
        for j in range(i + 1, len(perfs)):
            collab_pairs.append({
                "artist_a": perfs[i],
                "artist_b": perfs[j],
                "song_id":  song_id
            })

collab_df = pd.DataFrame(collab_pairs)
collab_df = collab_df.merge(
    all_works[["id", "release_date"]].rename(columns={"id": "song_id"}),
    on="song_id", how="left"
)
collab_df["release_date"] = pd.to_numeric(collab_df["release_date"], errors="coerce")

# Long format: each artist appears once per row as "artist_id"
collab_long = pd.concat([
    collab_df.rename(columns={"artist_a": "artist_id", "artist_b": "collaborator"}),
    collab_df.rename(columns={"artist_b": "artist_id", "artist_a": "collaborator"})
])[["artist_id", "collaborator", "song_id", "release_date"]]

collab_with_popular = (
    collab_long[collab_long["collaborator"].isin(popular_ids)]
    .groupby("artist_id")["collaborator"].nunique()
    .reset_index(name="collab_with_popular_count")
)
metrics = metrics.merge(collab_with_popular, on="artist_id", how="left")
metrics["collab_with_popular_count"] = metrics["collab_with_popular_count"].fillna(0)

# ── SCORE PARAM 5A: RECENT COLLAB COUNT ──────────────────────────────────────
# Unique collaborators in last RECENT_COLLAB years — are they actively networking?
recent_collab_counts = (
    collab_long[collab_long["release_date"] >= (CURRENT_YEAR - RECENT_COLLAB)]
    .groupby("artist_id")["collaborator"].nunique()
    .reset_index(name="recent_collab_count")
)
metrics = metrics.merge(recent_collab_counts, on="artist_id", how="left")
metrics["recent_collab_count"] = metrics["recent_collab_count"].fillna(0)

# ── SCORE PARAM 5B: COLLAB GROWTH RATE ───────────────────────────────────────
# Slope of cumulative unique collaborators over career
# Captures momentum trend — is the network expanding or flat?
yearly_collabs = (
    collab_long
    .groupby(["artist_id", "release_date"])["collaborator"]
    .nunique()
    .reset_index(name="new_collabs")
    .sort_values(["artist_id", "release_date"])
)
yearly_collabs["cumulative_collabs"] = (
    yearly_collabs.groupby("artist_id")["new_collabs"].cumsum()
)

def calc_slope(group):
    if len(group) < 2:
        return 0.0
    slope, *_ = linregress(group["release_date"], group["cumulative_collabs"])
    return slope

collab_growth = (
    yearly_collabs
    .groupby("artist_id")
    .apply(calc_slope)
    .reset_index(name="collab_growth_rate")
)
metrics = metrics.merge(collab_growth, on="artist_id", how="left")
metrics["collab_growth_rate"] = metrics["collab_growth_rate"].fillna(0)

# ── SCORE PARAM 6: RECENT INBOUND INFLUENCE (recency-adjusted) ────────────────
# Only count inbound influence edges where the CITING song is recent
# Corrects for the crowdsourced data lag (Sailor has inbound=0 without this fix)
recent_citing_ids = set(
    all_works[all_works["release_date"] >= (CURRENT_YEAR - RECENT_COLLAB)]["id"]
)
metrics["recent_inbound_influence"] = metrics["artist_id"].map(
    lambda aid: all_influence[
        (all_influence["target"].isin(work_ids_by_artist.get(aid, set()))) &
        (all_influence["source"].isin(recent_citing_ids))
    ].shape[0]
)

# ── SCORE PARAM 7: GENRE ALIGNMENT ───────────────────────────────────────────
# Uses GENRE_WINDOW (10yr) to identify hot genres — wider than RECENT_WINDOW
# so we get a meaningful top-5 set rather than just Oceanus Folk

recent_works_for_genres = all_works[
    all_works["release_date"] >= (CURRENT_YEAR - GENRE_WINDOW)
]
top_genres = set(
    recent_works_for_genres[recent_works_for_genres["notoriety_date"].notna()]
    .groupby("genre").size()
    .nlargest(5).index
)
print(f"\nTop 5 hot genres ({GENRE_WINDOW}yr window): {top_genres}")

genre_aligned = (
    artist_works_df[
        (artist_works_df["release_date"] >= (CURRENT_YEAR - RECENT_COLLAB)) &
        (artist_works_df["genre"].isin(top_genres))
    ]
    .groupby("artist_id").size()
    .reset_index(name="genre_alignment_score")
)
metrics = metrics.merge(genre_aligned, on="artist_id", how="left")
metrics["genre_alignment_score"] = metrics["genre_alignment_score"].fillna(0)

# ── SCORE PARAM 8: GENRE DIVERSITY ───────────────────────────────────────────
# Already in base metrics — no additional work needed

# ── SCORE PARAM 9: PRESTIGE SCORE ────────────────────────────────────────────
# Label prestige: worked with a label that has produced above-median notable works
label_works_df = pd.concat([
    recorded_by.merge(
        all_works[["id", "notable"]], left_on="source", right_on="id"
    )[["target", "notable"]],
    distributed.merge(
        all_works[["id", "notable"]], left_on="source", right_on="id"
    )[["target", "notable"]]
]).rename(columns={"target": "label_id"})

label_scores = (
    label_works_df[label_works_df["notable"] == True]
    .groupby("label_id").size()
    .reset_index(name="label_notable_count")
)
prestige_labels = set(
    label_scores[
        label_scores["label_notable_count"] >= label_scores["label_notable_count"].median()
    ]["label_id"]
)

artist_songs   = performer_of.rename(columns={"source": "artist_id", "target": "song_id"})
song_label_map = pd.concat([
    recorded_by.rename(columns={"source": "song_id", "target": "label_id"})[["song_id", "label_id"]],
    distributed.rename(columns={"source": "song_id", "target": "label_id"})[["song_id", "label_id"]]
])
artist_label_link = artist_songs.merge(song_label_map, on="song_id")

prestige_label_count = (
    artist_label_link[artist_label_link["label_id"].isin(prestige_labels)]
    .groupby("artist_id")["label_id"].nunique()
    .reset_index(name="prestige_label_count")
)
metrics = metrics.merge(prestige_label_count, on="artist_id", how="left")
metrics["prestige_label_count"] = metrics["prestige_label_count"].fillna(0)

# Group prestige: member of a group whose members have above-median notable works
group_member_scores = (
    member_of
    .merge(metrics[["artist_id", "notable_works"]], left_on="source", right_on="artist_id")
    .groupby("target")["notable_works"].sum()
    .reset_index(name="group_prestige")
)
prestige_groups = set(
    group_member_scores[
        group_member_scores["group_prestige"] >= group_member_scores["group_prestige"].median()
    ]["target"]
)
prestige_group_count = (
    member_of[member_of["target"].isin(prestige_groups)]
    .groupby("source")["target"].nunique()
    .reset_index(name="prestige_group_count")
    .rename(columns={"source": "artist_id"})
)
metrics = metrics.merge(prestige_group_count, on="artist_id", how="left")
metrics["prestige_group_count"] = metrics["prestige_group_count"].fillna(0)

metrics["prestige_score"] = metrics["prestige_label_count"] + metrics["prestige_group_count"]

# ── SCORE PARAM 10: INFLUENCED BY POPULAR ARTISTS ────────────────────────────
# Influence edges from this artist's songs toward charted (notable) songs
# Signals commercial awareness — they know what works in the market
charted_ids = set(all_works[all_works["notoriety_date"].notna()]["id"])

metrics["influenced_by_popular_score"] = metrics["artist_id"].map(
    lambda aid: all_influence[
        (all_influence["source"].isin(work_ids_by_artist.get(aid, set()))) &
        (all_influence["target"].isin(charted_ids))
    ].shape[0]
)

# ── SCORE PARAM 11: ROLE DIVERSITY ───────────────────────────────────────────
# 1 = performer only, 2 = performer + one other role, 3 = all three roles
# Used as colour encoding in Tableau DB5; also a minor score component
role_edges = edges[edges["Edge Type"].isin(["PerformerOf", "ComposerOf", "LyricistOf"])]
role_diversity = (
    role_edges.groupby("source")["Edge Type"].nunique()
    .reset_index(name="role_diversity")
    .rename(columns={"source": "artist_id"})
)
metrics = metrics.merge(role_diversity, on="artist_id", how="left")
metrics["role_diversity"] = metrics["role_diversity"].fillna(0)

# ── OCEANUS FOLK ALIGNMENT (for prediction dashboard) ────────────
# Separate from genre_alignment_score — this specifically measures how much an artist creates and draws from Oceanus Folk, used as the y-axis in the prediction quadrant
# Component 1: count of OF-genre works performed by this artist
of_ids = set(all_works[all_works["genre"] == "Oceanus Folk"]["id"])

of_genre_works = (
    artist_works_df[artist_works_df["genre"] == "Oceanus Folk"]
    .groupby("artist_id")
    .size()
    .reset_index(name="of_genre_works")
)
metrics = metrics.merge(of_genre_works, on="artist_id", how="left")
metrics["of_genre_works"] = metrics["of_genre_works"].fillna(0).astype(int)

# Component 2: influence edges from this artist's songs TO OF works
# (unchanged from before — how much they draw inspiration from OF)
metrics["of_influence_edges"] = metrics["artist_id"].map(
    lambda aid: all_influence[
        (all_influence["source"].isin(work_ids_by_artist.get(aid, set()))) &
        (all_influence["target"].isin(of_ids))
    ].shape[0]
)

# Combined score — simple sum of both components
# You can weight them differently if needed, but equal weight is defensible:
# producing OF music and being inspired by OF music are both valid alignment signals
metrics["of_alignment"] = metrics["of_genre_works"] + metrics["of_influence_edges"]

# Spot-check
print("=== OF ALIGNMENT CHECK ===")
check = metrics[metrics["artist_id"].isin([17255, 1304, 1843])][
    ["artist_id", "of_genre_works", "of_influence_edges", "of_alignment"]
]
print(check.to_string())

# ── ATTACH NAMES ──────────────────────────────────────────────────────────────
metrics = metrics.merge(
    persons[["id", "name", "stage_name"]],
    left_on="artist_id", right_on="id", how="left"
)

# Display name: stage name if available, otherwise real name
metrics["display_name"] = metrics["stage_name"].where(
    metrics["stage_name"].notna(), metrics["name"]
)

# ── APPLY HARD FILTERS ────────────────────────────────────────────────────────
rising_candidates = metrics[
    (metrics["is_performer"]    == True) &
    (metrics["is_recent_debut"] == True) &
    (metrics["name"].notna())             # exclude nameless nodes (data quality gap)
].copy()

print(f"\nCandidates after hard filters: {len(rising_candidates)}")
print(f"  - Excluded non-performers : {(metrics['is_performer'] == False).sum()}")
print(f"  - Excluded old debuts     : {(metrics['is_recent_debut'] == False).sum()}")
print(f"  - Excluded nameless nodes : {metrics['name'].isna().sum()}")

# ── NORMALISE COLLAB COMPONENTS FIRST ────────────────────────────────────────
# Combine recent_collab_count + collab_growth_rate into one collab_score (65/35)
# before the main normalisation pass
scaler_pre = MinMaxScaler()
rising_candidates[["recent_collab_norm_pre", "growth_norm_pre"]] = scaler_pre.fit_transform(
    rising_candidates[["recent_collab_count", "collab_growth_rate"]].fillna(0)
)
rising_candidates["collab_score"] = (
    rising_candidates["recent_collab_norm_pre"] * 0.65 +
    rising_candidates["growth_norm_pre"]         * 0.35
)

# ── SCORE WEIGHTS ─────────────────────────────────────────────────────────────
# Weights sum to 1.00
# notoriety_recency_score and collab_score are already 0–1, skip rescaling
score_components = {
    "recent_notable_works":        0.16,
    "outbound_influence":          0.14,
    "notoriety_recency_score":     0.13,   # already 0–1
    "collab_with_popular_count":   0.12,
    "collab_score":                0.10,   # already 0–1
    "recent_inbound_influence":    0.08,
    "genre_alignment_score":       0.08,
    "genre_diversity":             0.07,
    "prestige_score":              0.05,
    "influenced_by_popular_score": 0.04,
    "role_diversity":              0.03,
}

already_normalised = {"notoriety_recency_score", "collab_score"}
cols_to_scale      = [c for c in score_components if c not in already_normalised]

scaler = MinMaxScaler()
scaled = scaler.fit_transform(rising_candidates[cols_to_scale].fillna(0))
for i, col in enumerate(cols_to_scale):
    rising_candidates[f"{col}_norm"] = scaled[:, i]

for col in already_normalised:
    rising_candidates[f"{col}_norm"] = rising_candidates[col]

# Final score
rising_candidates["rising_star_score"] = sum(
    rising_candidates[f"{col}_norm"] * weight
    for col, weight in score_components.items()
)

# ── RANK ──────────────────────────────────────────────────────────────────────
rising_candidates["rank"] = rising_candidates["rising_star_score"].rank(
    ascending=False, method="min"
)

# ── VALIDATE: SAILOR SHIFT ────────────────────────────────────────────────────
sailor = rising_candidates[rising_candidates["name"] == "Sailor Shift"]
print("\n" + "=" * 60)
print("VALIDATION — SAILOR SHIFT")
print("=" * 60)
if len(sailor) > 0:
    print(f"Rank  : {int(sailor['rank'].values[0])} / {len(rising_candidates)}")
    print(f"Score : {sailor['rising_star_score'].values[0]:.4f}")
    print("\nComponent breakdown:")
    for col in score_components:
        norm_val = sailor[f"{col}_norm"].values[0]
        raw_val  = sailor[col].values[0]
        print(f"  {col:<35} raw={raw_val:>7.2f}  norm={norm_val:.3f}  contrib={norm_val * score_components[col]:.4f}")
else:
    print("Sailor Shift not found — check is_performer and is_recent_debut flags")
    print(metrics[metrics["name"] == "Sailor Shift"][
        ["name","is_performer","is_recent_debut","debut_year"]
    ].to_string())

# ── DEBUG: COMPONENT BREAKDOWN FOR TOP 10 ────────────────────────────────────
print("\n" + "=" * 60)
print("TOP 10 — COMPONENT SCORES")
print("=" * 60)
norm_cols = [f"{c}_norm" for c in score_components]
top10 = rising_candidates.sort_values("rising_star_score", ascending=False).head(10)
print(top10[["rank", "display_name", "debut_year", "rising_star_score"] + norm_cols].to_string())

# ── TOP 10 SUMMARY ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("TOP 10 RISING STARS — SUMMARY")
print("=" * 60)
summary_cols = [
    "rank", "display_name", "debut_year", "rising_star_score",
    "recent_notable_works", "outbound_influence", "collab_with_popular_count",
    "recent_collab_count", "of_alignment", "role_diversity", "genre_diversity"
]
print(top10[summary_cols].to_string())

# ── CLEAN DATATYPES BEFORE EXPORT ────────────────────────────────────────────

# Columns that should be whole-number years — convert float to nullable integer
year_cols = [
    "debut_year", "latest_year", "first_notoriety", "last_notoriety",
    "career_span", "notoriety_lag", "years_since_notoriety"
]
for col in year_cols:
    if col in metrics.columns:
        metrics[col] = pd.to_numeric(metrics[col], errors="coerce").astype("Int64")
    if col in rising_candidates.columns:
        rising_candidates[col] = pd.to_numeric(rising_candidates[col], errors="coerce").astype("Int64")

# Columns that should be whole-number counts
count_cols = [
    "total_works", "notable_works", "recent_notable_works",
    "outbound_influence", "inbound_influence", "recent_inbound_influence",
    "collab_with_popular_count", "recent_collab_count",
    "genre_diversity", "role_diversity", "of_alignment",
    "prestige_score", "prestige_label_count", "prestige_group_count",
    "influenced_by_popular_score", "genre_alignment_score"
]
for col in count_cols:
    if col in metrics.columns:
        metrics[col] = pd.to_numeric(metrics[col], errors="coerce").astype("Int64")
    if col in rising_candidates.columns:
        rising_candidates[col] = pd.to_numeric(rising_candidates[col], errors="coerce").astype("Int64")

# Score columns stay as float — round to 4 decimal places for clean display
score_cols = [
    "rising_star_score", "notoriety_recency_score", "collab_score",
    "collab_growth_rate"
]
for col in score_cols:
    if col in rising_candidates.columns:
        rising_candidates[col] = rising_candidates[col].round(4)

print("Datatypes cleaned.")
print(metrics[["debut_year", "notable_works", "genre_diversity"]].dtypes)

# ── EXPORT ALL CSVS ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EXPORTING CSVs")
print("=" * 60)

# 1. Full metrics — all artists, all fields (for Tableau data model)
metrics.to_csv("artist_metrics_full.csv", index=False)
print("  artist_metrics_full.csv")

# 2. Scored rising candidates — filtered, scored, ranked (for DB5)
rising_candidates.to_csv("rising_stars_scored.csv", index=False)
print("  rising_stars_scored.csv")

# 3. Yearly collab trajectory — one row per artist per year (for DB4 lines)
yearly_collabs_export = yearly_collabs.merge(
    persons[["id", "name", "stage_name"]], left_on="artist_id", right_on="id", how="left"
)
yearly_collabs_export["display_name"] = yearly_collabs_export["stage_name"].where(
    yearly_collabs_export["stage_name"].notna(), yearly_collabs_export["name"]
)
yearly_collabs_export.to_csv("yearly_collabs.csv", index=False)
print("  yearly_collabs.csv")

# 4. Influence edges enriched with source/target genre + year (for DB3)
influence_edges_export = all_influence.copy()
work_attrs = all_works[["id", "genre", "release_date"]].set_index("id")
influence_edges_export["source_genre"] = influence_edges_export["source"].map(
    lambda x: work_attrs["genre"].get(x)
)
influence_edges_export["source_year"]  = influence_edges_export["source"].map(
    lambda x: work_attrs["release_date"].get(x)
)
influence_edges_export["target_genre"] = influence_edges_export["target"].map(
    lambda x: work_attrs["genre"].get(x)
)
influence_edges_export.to_csv("influence_edges.csv", index=False)
print("  influence_edges.csv")

# 5. Sailor ego network nodes — artist IDs within 2 hops (for DB2)
import networkx as nx
G_collab = nx.from_pandas_edgelist(
    performer_of, source="source", target="target", create_using=nx.Graph()
)
if SAILOR_ID in G_collab:
    ego_1 = set(nx.ego_graph(G_collab, SAILOR_ID, radius=1).nodes())
    ego_2 = set(nx.ego_graph(G_collab, SAILOR_ID, radius=2).nodes())
    sailor_ego = metrics[metrics["artist_id"].isin(ego_2)].copy()
    sailor_ego["hop"] = sailor_ego["artist_id"].map(
        lambda x: 0 if x == SAILOR_ID else (1 if x in ego_1 else 2)
    )
    sailor_ego.to_csv("sailor_ego_nodes.csv", index=False)
    print(f"  sailor_ego_nodes.csv  ({len(sailor_ego)} nodes, depth=2)")

print("\nDone.")